In [1]:
# check. on human data structure
import pandas as pd
from pathlib import Path
df_human = pd.read_csv('raw-responses/barriers_human_final.csv')

# create a folder
output_dir = Path('barriers-zeroshot-working')
output_dir.mkdir(parents=True, exist_ok=True)

df_human.info()

# drop the following columns
df_human.drop(columns=['base_model', 'variant_id', 'model', 'bias_type', 'anchor_level', 'scenario'], inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         115 non-null    object 
 1   base_model     0 non-null      float64
 2   variant_id     0 non-null      float64
 3   model          0 non-null      float64
 4   barrier_id     115 non-null    int64  
 5   barrier_label  115 non-null    object 
 6   condition      115 non-null    object 
 7   source         115 non-null    object 
 8   scenario       115 non-null    object 
 9   bias_type      0 non-null      float64
 10  anchor_level   0 non-null      float64
 11  iteration      0 non-null      float64
 12  timestamp      0 non-null      float64
dtypes: float64(7), int64(1), object(5)
memory usage: 11.8+ KB


In [2]:
# take the new df_human here
df_human.info()

# change values of column 'condition' to be empty
df_human['condition'] = ''

# add a column 'base_model' with the value 'human' for all rows
df_human['base_model'] = 'human'

# deliverance
df_human['variant_id'] = df_human['row_id'].str.split('_').str[0]

# rename the column 'barrier_label' to 'official_label'
df_human.rename(columns={'barrier_label': 'official_label'}, inplace=True)

# save the new df_human to a csv file
df_human.to_csv(output_dir / 'barrier_human_final_clean.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         115 non-null    object 
 1   barrier_id     115 non-null    int64  
 2   barrier_label  115 non-null    object 
 3   condition      115 non-null    object 
 4   source         115 non-null    object 
 5   iteration      0 non-null      float64
 6   timestamp      0 non-null      float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.4+ KB


In [3]:
df_human_c = pd.read_csv('barriers-zeroshot-working/barrier_human_final_clean.csv')
df_human_c.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   row_id          115 non-null    object 
 1   barrier_id      115 non-null    int64  
 2   official_label  115 non-null    object 
 3   condition       0 non-null      float64
 4   source          115 non-null    object 
 5   iteration       0 non-null      float64
 6   timestamp       0 non-null      float64
 7   base_model      115 non-null    object 
 8   variant_id      115 non-null    object 
dtypes: float64(3), int64(1), object(5)
memory usage: 8.2+ KB


In [5]:
# llm data
df_llm = pd.read_csv('../21_zero-shot/21_zero-shot-raw/21_zeroshotBarrierSelectResponses.csv')
df_llm.info()

# count unique iterations
unique_iterations = df_llm['iteration'].nunique()
print(f"Number of unique iterations: {unique_iterations}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   condition           3000 non-null   object 
 1   anchor_type         0 non-null      float64
 2   anchor_level        0 non-null      float64
 3   row_id              3000 non-null   object 
 4   base_model          3000 non-null   object 
 5   variant_id          3000 non-null   object 
 6   model               3000 non-null   object 
 7   barrier_id          3000 non-null   int64  
 8   displayed_position  3000 non-null   int64  
 9   official_label      3000 non-null   object 
 10  model_label         3000 non-null   object 
 11  label_status        3000 non-null   object 
 12  is_hallucinated     3000 non-null   bool   
 13  iteration           3000 non-null   int64  
 14  timestamp           3000 non-null   object 
 15  attempt_id          3000 non-null   object 
 16  model_

In [7]:
# cell3: merging human and llm data
import pandas as pd

# load data
df_human = pd.read_csv('barriers-zeroshot-working/barrier_human_final_clean.csv')
df_llm = pd.read_csv('../21_zero-shot/21_zero-shot-raw/21_zeroshotBarrierSelectResponses.csv')

# create output directory
output_dir = Path('barriers-zeroshot-working')
output_dir.mkdir(parents=True, exist_ok=True)

# define source and condition
df_human['source'] = 'human'
df_llm['source'] = 'llm'

df_human['condition'] = 'zeroshot'
df_llm['condition'] = 'zeroshot'

# exact columns to keep
columns_to_keep = ['row_id', 'variant_id', 'base_model', 'source', 'model', 'barrier_id', 'official_label', 'label', 'iteration']


# add any missing columns
for column in columns_to_keep:
    if column not in df_human.columns: df_human[column] = pd.NA
    if column not in df_llm.columns: df_llm[column] = pd.NA

# combine the dataframes
df_combined = pd.concat([df_human[columns_to_keep], df_llm[columns_to_keep]], ignore_index=True)

# save combined dataframe
df_combined.to_csv(output_dir / 'barriers-humanllm-responses.csv', index=False)


In [8]:
df_comb = pd.read_csv('barriers-zeroshot-working/barriers-humanllm-responses.csv')